In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
)

TRANSACTION_SCHEMA = StructType([
    StructField("date", StringType(), True),

    StructField("city", StringType(), True),
    StructField("district", StringType(), True),
    StructField("business_district", StringType(), True),
    StructField("community", StringType(), True),

    StructField("layout", StringType(), True),
    StructField("orientation", StringType(), True),

    StructField("floor", StringType(), True),

    StructField("area_sqm", DoubleType(), True),

    StructField("listing_price_wan", DoubleType(), True),
    StructField("deal_price_wan", DoubleType(), True),
    StructField("deal_price_yuan", DoubleType(), True),

    StructField("deal_cycle", StringType(), True),
])

In [ ]:
from pyspark.sql import functions as F

history_df = (spark.read
            .format("csv")
            .option("header", "true")
            .schema(TRANSACTION_SCHEMA)
            .option("quote", '"')
            .option("escape", '"')
            .option("recursiveFileLookup", "true")
            .load("s3://housing-intelligence-data/raw_data/sales_data/shell_data/")
            .withColumn(
                "transaction_year",
                F.split(F.col("date"), "\\.")[0]
            ))

In [ ]:
(history_df.write.format("delta")
                .mode("overwrite")
                .partitionBy("transaction_year")
                .saveAsTable("housing_intelligence.raw.sales_data"))